Imports and Initialization

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (classification_report, confusion_matrix, 
                             roc_auc_score, roc_curve)

# Styling and directory setups
sns.set_theme(style="whitegrid")
os.makedirs('charts', exist_ok=True)
print("Libraries loaded. Workspace ready for HR Analytics pipeline!")

Libraries loaded. Workspace ready for HR Analytics pipeline!


Task 1 — Data Loading & Exploration

In [2]:
# Load dataset (adjust filename to match your local filename if needed)
df = pd.read_csv('WA_Fn-UseC_-HR-Employee-Attrition.csv')

print("--- Dataset Dimensions ---")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}\n")

print("--- First 3 Rows Preview ---")
display(df.head(3))

# Calculate Attrition Rate
attrition_counts = df['Attrition'].value_counts()
attrition_rate = (attrition_counts['Yes'] / df.shape[0]) * 100

print("\n--- Attrition Target Distribution ---")
print(f"Employees Stayed (No): {attrition_counts['No']}")
print(f"Employees Left (Yes): {attrition_counts['Yes']}")
print(f"Base Attrition Rate: {attrition_rate:.2f}%")

# Count structural datatypes
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f"\nNumeric features count: {len(numeric_cols)}")
print(f"Categorical features count: {len(categorical_cols)}")

print("\n--- Task 1 Observation ---")
print("Observation: The dataset exhibits a significant class imbalance. Roughly 16.12% of the employees ")
print("have left, meaning the baseline model could achieve 84% accuracy by simply guessing 'No' for everyone.")
print("We must adjust model class weights to account for this skew.")

--- Dataset Dimensions ---
Rows: 1470, Columns: 35

--- First 3 Rows Preview ---


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0



--- Attrition Target Distribution ---
Employees Stayed (No): 1233
Employees Left (Yes): 237
Base Attrition Rate: 16.12%

Numeric features count: 26
Categorical features count: 9

--- Task 1 Observation ---
Observation: The dataset exhibits a significant class imbalance. Roughly 16.12% of the employees 
have left, meaning the baseline model could achieve 84% accuracy by simply guessing 'No' for everyone.
We must adjust model class weights to account for this skew.


C:\Users\Asus\AppData\Local\Temp\ipykernel_8124\584099526.py:21: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=['object']).columns.tolist()


Task 2 — Data Cleaning & Preprocessing

In [3]:
# 1. Verify missing values
print("Total Missing Values:", df.isnull().sum().sum())

# 2. Drop low-variance/irrelevant columns
cols_to_drop = ['EmployeeNumber', 'Over18', 'StandardHours', 'EmployeeCount']
df_cleaned = df.drop(columns=[col for col in cols_to_drop if col in df.columns])

# 3. Target mapping
df_cleaned['Attrition'] = df_cleaned['Attrition'].map({'Yes': 1, 'No': 0})

# 4. Separate target and features
X = df_cleaned.drop(columns=['Attrition'])
y = df_cleaned['Attrition']

# Identify remaining categorical columns for encoding (Updated to support Pandas 3/4 string transitions)
cat_features = X.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
print(f"Categorical features detected for encoding: {cat_features}")

# 5. One-Hot Encoding (Enforce integer 1/0 formatting explicitly)
X_encoded = pd.get_dummies(X, columns=cat_features, drop_first=True, dtype=int)

# 6. Feature Scaling
scaler = StandardScaler()
num_features = X.select_dtypes(include=[np.number]).columns.tolist()
X_scaled = X_encoded.copy()
X_scaled[num_features] = scaler.fit_transform(X_encoded[num_features])

print(f"\nFinal feature dimensions after cleaning & encoding: {X_scaled.shape}")

Total Missing Values: 0
Categorical features detected for encoding: ['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime']

Final feature dimensions after cleaning & encoding: (1470, 44)


Task 3 — Exploratory Data Analysis & Feature Mining